---
---
# TUTORIAL 7: CONFIDENCE INTERVALS

In this tutorial you'll gain further exposure to confidence intervals. In particular, we'll elaborate on the term *confidence*, we'll gain practice with *percentile-method* confidence intervals, and you'll explore the impact of *confidence level* and *sample size* on confidence intervals.

---
---
## Understanding Confidence

A **confidence interval** is an interval of numbers that we believe contains the true population attribute. Confidence intervals are reported alongside sample estimates to communicate our uncertainty in the estimate and to provide a broader sense of what the population attribute value might be. 

### What does it mean to be confident?

When we construct a ${C\%}$ confidence interval, and we say we're ${C\%}$ confident it contains the true population attribute, what we mean is:

> *$C\%$ of confidence intervals constructed in this way should cover the true population attribute*.


For instance, suppose we have 20 samples, each producing a 95% confidence interval.

<div style="text-align: center;">
<img src="https://raw.githubusercontent.com/ntstevens/DATSC101/refs/heads/main/images/coverage1.svg" alt="repeated_resamples" width="300"/>
</div>

Because these are 95% confidence intervals, we expect 95% of them (in this case, 19 out of 20) to contain, or **cover**, the true population attribute. 

<div style="text-align: center;">
<img src="https://raw.githubusercontent.com/ntstevens/DATSC101/refs/heads/main/images/coverage2.svg" alt="repeated_resamples" width="300"/>
</div>

This is what we mean by *confidence*. We have confidence in the procedure that produces the intervals. We trust in the process.

### What *doesn't* it mean to be confident?

The following is a common misconception:

> *I calculated a 95% confidence interval. The probability my interval contains the true population value is 0.95.*

This is incorrect, false, and not how confidence intervals work. 

* Remember what probability quantifies: it quantifies uncertainty in a *random process*.
* Once you've obtained *your* interval, there is nothing random about it. 
* Your interval either contains the population value or not. You don't know one way or the other, but this outcome is not random.
* The probability your interval contains the population value is therefore 0 or 1.

This line of thinking arises in other contexts as well. It's worthwhile to review some other examples to reinforce why the statement above is inaccurate.

#### Example 1: Lottery Tickets

Consider scratch lottery tickets like the [Bingo](https://www.olg.ca/en/instants/play-bingo-multiplier.html) ones pictured below.

<div style="text-align: center;">
<img src="https://raw.githubusercontent.com/ntstevens/DATSC101/refs/heads/main/images/Bingo.png" alt="Bingo" width="300"/>
</div>

The OLG (Ontario Lottery and Gaming Corporation) website states that this game has an "overall odds of winning: 1 in 3.59." 

These odds can be translated into a probability as follows:
* If the odds of winning are 1 in 3.59, the probability of winning is 1/3.59 = 0.2785515.

But what does this mean for you, the player? 

* If you've purchased one of these cards, what is the probability *your* card is a winner?
* That probability is 0 or 1. *Your* card is a winner or it isn't. There is no randomness there. 
* The statement *"the probability of winning is 0.2785515"* applies to the random process that generates the cards, not your card in particular.

#### Example 2: Pokémon Cards

The Pokémon Trading Card Game (TCG) is a collectible and competitive card game where players act as Pokémon Trainers, battling each other using custom decks. Some cards are rarer (i.e., harded to get) than others. This makes them potentially very lucrative. 

Recently, a new expansion in the *Mega Evolution Series* of the Pokémon TCG was released: *Chaos Rising*.

<div style="text-align: center;">
<img src="https://www.kingdomtitans.ca/cdn/shop/files/Pokemon-TCG-Mega-Evolution-Chaos-Rising-Booster-Wrap-Mega-Greninja_EN-780x1426-5978221.png" alt="Pokemon" width="300"/>
</div>

One card in particular is highly soughtafter: **Mega Greninja ex**

<div style="text-align: center;">
<img src="https://www.cardrake.com/_next/image?url=https%3A%2F%2Ftcgplayer-cdn.tcgplayer.com%2Fproduct%2F693518_in_1000x1000.jpg&w=640&q=75" alt="Mega Greninja ex" width="300"/>
</div>

**Mega Greninja ex** is a "mega hyper rare" card with a [pull rate](https://pullrates.com/set/chaos-rising) of 1 in 956. This means that 1 pack in 956 contains this card (on average). In other words, the probability of obtaining a pack containing  Mega Greninja ex is 1/956 = 0.000104.

But what does this mean for you, the collector? 

* If you've purchased one of these packs, what is the probability *your* pack contains Mega Greninja ex?
* That probability is 0 or 1. *Your* pack either includes this card or not. There is no randomness there. 
* The statement *"the probability pulling Mega Greninja ex is 0.000104"* applies to the random process that fills the packs with cards, not your pack in particular.

[Let's open some packs and see if we get lucky...](https://pullrates.com/set/chaos-rising)



---
---

## Canadian Student Alcohol & Drugs Survey

<div style="text-align: center;">
<img src="https://www.csads.ca/Info/Content/img/CTADS_LOGO_FINAL_ENG.png" alt="CSADS" width="600"/>
</div>

Recall, the [Canadian Student Alcohol and Drugs Survey](https://www.csads.ca/Info/en/Home) (CSADS) is a survey conducted by Health Canada every two years to track Canadian trends in student alcohol and drug use. The survey asks students from grades 7 to 12 about their alcohol and drug use, as well as demographic information and other
questions related to their social and psychological context.

In the 2023-2024 survey cycle, $n=51,844$ students responded. The results are stored in the csv file `202324-csads-pumf-data.csv`. Among the many questions asked on this survey, one question (`ALC_050`) concerned *binge drinking*: drinking 5 or more drinks in a single occasion. The question was as follows:

> In the last 12 months, how often did you have 5 or more drinks of alcohol on one occasion?

And the responses were recorded as follows:

```
1 = I have never had 5 or more drinks of alcohol on one occasion
2 = I have not done this in the last 12 months
3 = Less than once a month
4 = Once a month
5 = 2 or 3 times a month
6 = Once a week
7 = 2 to 5 times a week
8 = Daily or almost daily
9 = I do not know
96 = Valid Skip (where ALC_010 = 2 or 99)
99 = Not Stated
```

In what follows, we'll estimate the proportion of students who've binge drank in the last year, and construct confidence intervals for this quantity. In doing so, we will also explore various characteristics of confidence intervals.

To do this, we will early on define a new boolean variable called `binge` that's `True` if a student answers 3-8 on `ALC_050` and `False` otherwise.

In [94]:
# Load the necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px

In [95]:
# Read in the data
csads = pd.read_csv("https://raw.githubusercontent.com/ntstevens/DATSC101/refs/heads/main/data/202324-csads-pumf-data.csv")
n = len(csads)

In [96]:
# Construct new `binge` variable
binge = (csads["ALC_050"] >= 3) & (csads["ALC_050"] <= 8)

---
---

### 1. Estimate the proportion of Grade 7-12 students in Canada who've binge drank in the past year.


In [ ]:
# Calculate the binge rate on the original data


### 2. Calculate a 95% percentile-method confidence interval for the proportion of Grade 7-12 students in Canada who've binge drank in the past year.

In [76]:
# Take B=10000 bootstrap resamples and calculate the binge rate on each


In [67]:
# 95% confidence interval
ci_95 = ____
print(ci_95)

### 3. What do you conclude about the prevalence bringe of drinking among Canadian students in grades 7-12?

### 4. Impact of Confidence Level

Repeat the confidence interval calculation above, but instead for 80%, 90%, and 99% confidence levels. Summarize the impact the confidence level has on the width of the interval.

In [ ]:
# 80% confidence interval
ci_80 = ____
print(ci_80)

In [ ]:
# 90% confidence interval
ci_90 = ____
print(ci_99)

In [ ]:
# 99% confidence interval
ci_99 = ____
print(ci_99)

In [ ]:
# Summarize widths and confidence levels
widths = [np.diff(ci_80),
          np.diff(ci_90),
          np.diff(ci_95),
          np.diff(ci_99)]
pd.DataFrame({"Conf Level":["80%", "90%", "95%", "99%"],
              "CI Width":widths})

### 5. Impact of Sample Size

Repeat the 95% confidence interval calculation, but instead pretend your sample was sized $n=100$, $n=1000$, $n=10000$. Summarize the impact sample size on the width of the interval.

**n=100**

In [79]:
# Use the following sample as if it was all the data we had
S_100 = binge.sample(n=100) 

In [ ]:
# Perform bootstrap 


# Calculate 95% confidence interval
ci_n100 = ____

**n=1000**

In [85]:
# Use the following sample as if it was all the data we had
S_1000 = binge.sample(n=1000) 

In [ ]:
# Perform bootstrap 


# Calculate 95% confidence interval
ci_n1000 = ____

**n=10000**

In [87]:
# Use the following sample as if it was all the data we had
S_10000 = binge.sample(n=10000) 

In [ ]:
# Perform bootstrap 


# Calculate 95% confidence interval
ci_n10000 = ____

In [ ]:
# Summarize widths and sample sizes
widths = [np.diff(ci_n100),
          np.diff(ci_n1000),
          np.diff(ci_n10000),
          np.diff(ci_95)]
pd.DataFrame({"Sample Size":["100", "1000", "10000", n],
              "CI Width":widths})